# Plot Paper Figures for LAMAR-m6A

This notebook is the GitHub-friendly plotting version of `paperfig-new.ipynb`.
It reads standardized outputs from the previous pipeline steps and writes final figures under `outputs/figures/`.

No training or prediction is performed here. Run the data, fine-tuning, evaluation, baseline, mutation, and SHAP notebooks first.

## 1. Project Root

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "lamar_m6a").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "lamar_m6a").exists():
    raise RuntimeError("Cannot find project root containing lamar_m6a/. Run this notebook inside the project directory.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## 2. Imports and Style

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import TwoSlopeNorm
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

try:
    import seaborn as sns
except ImportError:
    sns = None

try:
    import logomaker
except ImportError:
    logomaker = None

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 12
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

CELL_LINES = ["a549", "cd8t", "hek293", "hela", "molm13"]
CELL_LABELS = {"a549": "A549", "cd8t": "CD8T", "hek293": "HEK293", "hela": "HeLa", "molm13": "MOLM13"}
CELL_COLORS = ["#90CAF9", "#FFAB91", "#A5D6A7", "#CE93D8", "#FFE082"]
MODEL_COLORS = {"LAMAR-M6A": "#2F6FDB", "deepSRAMP": "#D64B4B", "DeepPromise": "#4C9A4C"}

FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MODE = "full"
LENGTH = 801
SPLIT = "test"
UNIFIED_MODEL_CELL = "combined"
INDEPENDENT_RUN_NAME = "combined_full_801"
SHAP_ANALYSIS_NAME = f"{UNIFIED_MODEL_CELL}_{LENGTH}"
MUTATION_ANALYSIS_PREFIX = f"{UNIFIED_MODEL_CELL}_model_on_cell5_{LENGTH}"

FIGURE_DIR

## 3. Helper Functions

In [ ]:
def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def pick_column(df, candidates):
    lower = {str(c).lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lower:
            return lower[name.lower()]
    raise KeyError(f"Missing one of columns: {candidates}. Existing columns: {list(df.columns)}")


def read_prediction_csv(path):
    df = pd.read_csv(path)
    label_col = pick_column(df, ["label", "labels", "y_true", "target"])
    pred_col = pick_column(df, ["predict", "predict_label", "pred_label", "y_pred"])
    prob_col = pick_column(df, ["predict_probs", "predict_prob", "pred_prob", "probability", "score"])
    out = df.copy()
    out["label"] = out[label_col].astype(int)
    out["predict"] = out[pred_col].astype(int)
    out["predict_probs"] = out[prob_col].astype(float)
    return out


def compute_metrics_from_prediction_df(df):
    y_true = df["label"].astype(int).to_numpy()
    y_pred = df["predict"].astype(int).to_numpy()
    y_prob = df["predict_probs"].astype(float).to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_prob)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": auc(recall_curve, precision_curve),
        "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "MCC": matthews_corrcoef(y_true, y_pred),
    }


def save_figure(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=600, bbox_inches="tight")
    print(f"saved: {path}")


def normalize_dataset_name(value):
    name = Path(str(value)).stem
    for suffix in ["_pred", "__merged", "_merged", ".csv"]:
        if name.endswith(suffix):
            name = name[: -len(suffix)]
    replacements = {
        "CM2005.1_miCLIP": "CM2005_miCLIP",
        "A549_m6A-seq_with_improved_protocol": "A549_m6A-seq",
        "Mouse_Embryonic_Fibroblasts_(MEFs)_miCLIP": "MEF_miCLIP",
        "Mouse_Embryonic_Fibroblasts_miCLIP": "MEF_miCLIP",
        "Brain_m6A-seq_with_improved_protocol": "Brain_m6A-seq",
        "dorsal_root_ganglion?(DRG)_m6A-CLIP-seq": "DRG_m6A-CLIP-seq",
    }
    name = name.replace(" ", "_").rstrip("_")
    for old, new in replacements.items():
        name = name.replace(old, new)
    return name


def prediction_path(model_cell, target_cell, length=LENGTH, split=SPLIT, mode=MODE):
    return PROJECT_ROOT / "outputs" / "predictions" / "cross_cell" / mode / f"model-{model_cell}_target-{target_cell}_{length}_{split}_pred.csv"


def cross_cell_metrics_path(length=LENGTH, split=SPLIT, mode=MODE):
    return PROJECT_ROOT / "outputs" / "metrics" / "cross_cell" / mode / f"cross_cell_{length}_{split}.csv"


def independent_metrics_path(run_name=INDEPENDENT_RUN_NAME, length=LENGTH, mode=MODE):
    return PROJECT_ROOT / "outputs" / "metrics" / "independent_atlas" / mode / f"{run_name}_{length}_metrics.csv"


def baseline_metrics_path(species):
    return PROJECT_ROOT / "outputs" / "metrics" / "baselines" / f"{species}_baseline_metrics.csv"


## 4. Figure 2A: Internal Test Metrics Across Five Cell Lines

Inputs are same-cell prediction CSVs produced by `evaluate_cross_cell.py`.

In [ ]:
fig2_dir = ensure_dir(FIGURE_DIR / "figure2")

same_cell_predictions = {cell: prediction_path(cell, cell) for cell in CELL_LINES}
for cell, path in same_cell_predictions.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing same-cell prediction for {cell}: {path}")

metrics_rows = []
for cell, path in same_cell_predictions.items():
    metrics = compute_metrics_from_prediction_df(read_prediction_csv(path))
    metrics["cell"] = CELL_LABELS[cell]
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows).set_index("cell")
metric_names = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "Specificity", "MCC"]

x = np.arange(len(metric_names))
bar_width = 0.8 / len(CELL_LINES)
fig, ax = plt.subplots(figsize=(8, 5))
for i, cell in enumerate(CELL_LINES):
    label = CELL_LABELS[cell]
    ax.bar(x + i * bar_width, metrics_df.loc[label, metric_names], bar_width, label=label, color=CELL_COLORS[i])

ax.set_ylabel("Value")
ax.set_ylim(0, 1.1)
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_xticks(x + bar_width * (len(CELL_LINES) - 1) / 2)
ax.set_xticklabels(metric_names, rotation=30, ha="right")
ax.legend(frameon=False, ncol=3)
fig.tight_layout()
save_figure(fig, fig2_dir / "fig2A_internal_metrics.png")
metrics_df.to_csv(fig2_dir / "fig2A_internal_metrics.csv")
metrics_df

## 5. Figure 2B and 2C: PR and ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for i, cell in enumerate(CELL_LINES):
    df = read_prediction_csv(same_cell_predictions[cell])
    precision, recall, _ = precision_recall_curve(df["label"], df["predict_probs"])
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, color=CELL_COLORS[i], lw=1.5, label=f"{CELL_LABELS[cell]} (AUC={pr_auc:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower left", frameon=False, fontsize=10)
fig.tight_layout()
save_figure(fig, fig2_dir / "fig2B_precision_recall.png")

fig, ax = plt.subplots(figsize=(5, 5))
for i, cell in enumerate(CELL_LINES):
    df = read_prediction_csv(same_cell_predictions[cell])
    fpr, tpr, _ = roc_curve(df["label"], df["predict_probs"])
    roc_auc = roc_auc_score(df["label"], df["predict_probs"])
    ax.plot(fpr, tpr, color=CELL_COLORS[i], lw=1.5, label=f"{CELL_LABELS[cell]} (AUC={roc_auc:.3f})")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower right", frameon=False, fontsize=10)
fig.tight_layout()
save_figure(fig, fig2_dir / "fig2C_roc.png")

## 6. Figure 2D: Cross-Cell PR-AUC

This uses `outputs/metrics/cross_cell/{mode}/cross_cell_{length}_{split}.csv` if available.

In [ ]:
metrics_path = cross_cell_metrics_path()
if metrics_path.exists():
    cross_df = pd.read_csv(metrics_path)
else:
    rows = []
    for model_cell in CELL_LINES:
        for target_cell in CELL_LINES:
            path = prediction_path(model_cell, target_cell)
            if path.exists():
                metrics = compute_metrics_from_prediction_df(read_prediction_csv(path))
                rows.append({"model_cell": model_cell, "target_cell": target_cell, "pr_auc": metrics["PR_AUC"]})
    cross_df = pd.DataFrame(rows)

if cross_df.empty:
    raise FileNotFoundError("No cross-cell metrics or prediction files were found.")

cross_df["model_cell"] = cross_df["model_cell"].astype(str).str.lower()
cross_df["target_cell"] = cross_df["target_cell"].astype(str).str.lower()
pr_col = "pr_auc" if "pr_auc" in cross_df.columns else "PR_AUC"
heat = cross_df.pivot(index="model_cell", columns="target_cell", values=pr_col).reindex(index=CELL_LINES, columns=CELL_LINES)
heat.index = [CELL_LABELS[c] for c in CELL_LINES]
heat.columns = [CELL_LABELS[c] for c in CELL_LINES]

fig, ax = plt.subplots(figsize=(5.5, 4.5))
if sns is not None:
    sns.heatmap(heat, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0, vmax=1, ax=ax, cbar_kws={"label": "PR-AUC"})
else:
    im = ax.imshow(heat.to_numpy(), vmin=0, vmax=1, cmap="YlGnBu")
    fig.colorbar(im, ax=ax, label="PR-AUC")
    ax.set_xticks(range(len(heat.columns)), heat.columns)
    ax.set_yticks(range(len(heat.index)), heat.index)
    for r in range(heat.shape[0]):
        for c in range(heat.shape[1]):
            ax.text(c, r, f"{heat.iloc[r, c]:.3f}", ha="center", va="center")
ax.set_xlabel("Target cell line")
ax.set_ylabel("Fine-tuned model")
fig.tight_layout()
save_figure(fig, fig2_dir / "fig2D_cross_cell_pr_auc.png")
heat.to_csv(fig2_dir / "fig2D_cross_cell_pr_auc.csv")
heat

## 7. Figure 3: Independent m6A-Atlas Model Comparison

Inputs:

- LAMAR-M6A: `outputs/metrics/independent_atlas/{mode}/{run_name}_{length}_metrics.csv`
- Baselines: `outputs/metrics/baselines/{species}_baseline_metrics.csv`

Run the baseline collection notebook first for each species.

In [ ]:
fig3_dir = ensure_dir(FIGURE_DIR / "figure3")

def load_lamar_independent_metrics(run_name=INDEPENDENT_RUN_NAME):
    path = independent_metrics_path(run_name=run_name)
    if not path.exists():
        raise FileNotFoundError(f"Missing LAMAR independent metrics: {path}")
    df = pd.read_csv(path)
    df = df.copy()
    df["model"] = "LAMAR-M6A"
    if "dataset" not in df.columns:
        if "input_csv" in df.columns:
            df["dataset"] = df["input_csv"].map(normalize_dataset_name)
        else:
            df["dataset"] = df.index.map(lambda i: f"dataset_{i}")
    df["dataset"] = df["dataset"].map(normalize_dataset_name)
    return df


def load_baseline_metrics(species):
    path = baseline_metrics_path(species)
    if not path.exists():
        raise FileNotFoundError(f"Missing baseline metrics: {path}")
    df = pd.read_csv(path).copy()
    df["species"] = species
    df["dataset"] = df["dataset"].map(normalize_dataset_name)
    return df


def plot_species_radar(species, lamar_df, baseline_df, metric="pr_auc"):
    combined = pd.concat([
        lamar_df.assign(species=species),
        baseline_df,
    ], ignore_index=True)
    combined = combined[combined[metric].notna()].copy()
    tools = ["LAMAR-M6A", "deepSRAMP", "DeepPromise"]
    comparison = combined.pivot_table(index="dataset", columns="model", values=metric, aggfunc="first")
    comparison = comparison[[tool for tool in tools if tool in comparison.columns]].dropna(how="any")
    if comparison.empty:
        raise ValueError(f"No overlapping datasets among tools for species={species}")
    comparison["Difference"] = comparison["LAMAR-M6A"] - comparison[[c for c in comparison.columns if c != "LAMAR-M6A"]].max(axis=1)
    comparison = pd.concat([
        comparison[comparison["Difference"] > 0].sort_values("LAMAR-M6A", ascending=False),
        comparison[comparison["Difference"] <= 0].sort_values("LAMAR-M6A", ascending=True),
    ])
    classes = comparison.index.tolist()
    angles = np.linspace(0, 2 * np.pi, len(classes), endpoint=False).tolist()
    angles += angles[:1]
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, polar=True)
    for tool in tools:
        if tool not in comparison.columns:
            continue
        values = comparison[tool].tolist() + [comparison[tool].iloc[0]]
        color = MODEL_COLORS.get(tool, "black")
        linestyle = "solid" if tool == "LAMAR-M6A" else ("dashed" if tool == "deepSRAMP" else "dotted")
        ax.fill(angles, values, alpha=0.18, color=color)
        ax.plot(angles, values, color=color, linestyle=linestyle, linewidth=1.8, label=tool)
        ax.scatter(angles[:-1], values[:-1], color=color, s=12, zorder=5)
    ax.set_rgrids(np.linspace(0, 1, 6), labels=[f"{v:.1f}" for v in np.linspace(0, 1, 6)], angle=0, fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([])
    ax.yaxis.grid(alpha=0.18)
    ax.xaxis.grid(alpha=0.18)
    for i, angle in enumerate(angles[:-1]):
        rot = np.rad2deg(angle)
        if 90 < rot <= 270:
            rot += 180
        ax.text(angle, 1.35, classes[i], ha="center", va="center", rotation=rot, rotation_mode="anchor", fontsize=7)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False)
    fig.tight_layout()
    save_figure(fig, fig3_dir / f"fig3_{species}_radar_pr_auc.png")
    comparison.to_csv(fig3_dir / f"fig3_{species}_comparison_pr_auc.csv")
    return comparison

lamar_metrics = load_lamar_independent_metrics()
comparisons = {}
for species in ["human", "mouse", "rat"]:
    baseline_df = load_baseline_metrics(species)
    comparisons[species] = plot_species_radar(species, lamar_metrics, baseline_df)
comparisons["human"].head()

## 8. Figure 4A and 4B: SHAP Summary Plots

Inputs are `.pkl` files generated by `run_shap_analysis_notebook.ipynb`.

In [ ]:
fig4_dir = ensure_dir(FIGURE_DIR / "figure4")

SHAP_DIR = PROJECT_ROOT / "outputs" / "interpretability" / "shap" / MODE / SHAP_ANALYSIS_NAME
LABEL1_SHAP_PKL = SHAP_DIR / "label1_test_500.pkl"
LABEL0_SHAP_PKL = SHAP_DIR / "label0_test_500.pkl"
MODEL_MAX_LENGTH = LENGTH + 2

if not LABEL1_SHAP_PKL.exists() or not LABEL0_SHAP_PKL.exists():
    raise FileNotFoundError(f"Missing SHAP pkl files: {LABEL1_SHAP_PKL}, {LABEL0_SHAP_PKL}")

with LABEL1_SHAP_PKL.open("rb") as f:
    label1_shap = pickle.load(f)
with LABEL0_SHAP_PKL.open("rb") as f:
    label0_shap = pickle.load(f)


def compute_mean_shap(shap_values_list, max_length):
    padded = []
    for values in shap_values_list:
        arr = np.asarray(values).squeeze()
        tmp = np.zeros(max_length)
        tmp[: min(len(arr), max_length)] = arr[:max_length]
        padded.append(tmp)
    return np.mean(padded, axis=0)

mean_label1 = compute_mean_shap(label1_shap, MODEL_MAX_LENGTH)[1:-1]
mean_label0 = compute_mean_shap(label0_shap, MODEL_MAX_LENGTH)[1:-1]
rel_x = np.arange(LENGTH) - (LENGTH // 2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rel_x, np.abs(mean_label1), color="coral", label="m6A-positive", linewidth=1)
ax.plot(rel_x, np.abs(mean_label0), color="skyblue", label="m6A-negative", linewidth=1)
ax.set_ylabel("Absolute Mean SHAP Value")
ax.set_xlabel("Position relative to candidate A")
ax.set_xlim(rel_x.min() - 10, rel_x.max() + 10)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
save_figure(fig, fig4_dir / "fig4A_shap_global.png")

window = 25
center = LENGTH // 2
idx = np.arange(center - window, center + window + 1)
x_rel = idx - center
y1 = np.abs(mean_label1[idx])
y0 = np.abs(mean_label0[idx])
bar_width = 0.4
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x_rel - bar_width / 2, y1, width=bar_width, color="coral", label="m6A-positive")
ax.bar(x_rel + bar_width / 2, y0, width=bar_width, color="skyblue", label="m6A-negative")
ax.set_ylabel("Absolute Mean SHAP Value")
ax.set_xlabel("Position relative to candidate A")
ax.set_xticks(x_rel)
ax.set_xticklabels(x_rel, rotation=90)
for pos, val in zip(x_rel[np.argsort(y1)[-5:]], y1[np.argsort(y1)[-5:]]):
    ax.annotate(str(pos), xy=(pos - bar_width / 2, val), xytext=(pos - bar_width / 2, val + max(y1.max(), 1e-6) * 0.08),
                ha="center", fontsize=9, arrowprops=dict(arrowstyle="->", color="red", lw=0.8))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=2, frameon=False)
fig.tight_layout()
save_figure(fig, fig4_dir / "fig4B_shap_center_51nt.png")

## 9. Figure 4C and Supplementary Figure S2: Mutation Effect Heatmaps

Input is the predicted mutation CSV generated by `run_mutation_analysis_notebook.ipynb`.

In [ ]:
MUTATION_CSV = PROJECT_ROOT / "outputs" / "interpretability" / "mutation" / MODE / f"{MUTATION_ANALYSIS_PREFIX}_label1_100_predict_linear.csv"
if not MUTATION_CSV.exists():
    fallback = PROJECT_ROOT / "outputs" / "interpretability" / "mutation" / "cell5_mutated_sequences_801_label1_100_predict_linear.csv"
    MUTATION_CSV = fallback if fallback.exists() else MUTATION_CSV
if not MUTATION_CSV.exists():
    raise FileNotFoundError(f"Missing mutation prediction CSV: {MUTATION_CSV}")

mut_df = pd.read_csv(MUTATION_CSV)
required = ["seq", "original_seq", "is_original", "position", "mutated_base"]
for col in required:
    if col not in mut_df.columns:
        raise ValueError(f"Missing required column in mutation CSV: {col}")
prob_col = pick_column(mut_df, ["predict_probs", "predict_prob", "probability", "score"])
mut_df["predict_probs"] = mut_df[prob_col].astype(float)
mut_df["is_original"] = mut_df["is_original"].astype(int)
original_probs = dict(zip(mut_df.loc[mut_df["is_original"] == 1, "original_seq"], mut_df.loc[mut_df["is_original"] == 1, "predict_probs"]))
mut_df["original_predict"] = mut_df["original_seq"].map(original_probs)
mut_df["Delta_pred_prob"] = np.where(mut_df["is_original"] == 1, 0.0, mut_df["predict_probs"] - mut_df["original_predict"])
mutation_only = mut_df[(mut_df["is_original"] == 0) & mut_df["Delta_pred_prob"].notna()].copy()
mutation_only["position0"] = mutation_only["position"].astype(int) - 1

pivot = mutation_only.pivot_table(values="Delta_pred_prob", index="mutated_base", columns="position0", aggfunc="mean")
pivot = pivot.reindex(index=["A", "C", "G", "T"])
pivot.to_csv(fig4_dir / "mutation_effect_by_position.csv")

v_bound = float(np.nanmax(np.abs(pivot.to_numpy())))
fig, ax = plt.subplots(figsize=(12, 2.6))
if sns is not None:
    sns.heatmap(pivot, cmap="coolwarm", center=0, vmin=-v_bound, vmax=v_bound, ax=ax, cbar_kws={"label": "Delta probability"})
else:
    im = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-v_bound, vmax=v_bound)
    fig.colorbar(im, ax=ax, label="Delta probability")
    ax.set_yticks(range(4), ["A", "C", "G", "T"])
ax.set_xlabel("Position")
ax.set_ylabel("Mutated base")
fig.tight_layout()
save_figure(fig, fig4_dir / "figS2_mutation_global_heatmap.png")

center = LENGTH // 2
window = 25
center_cols = [c for c in pivot.columns if center - window <= c <= center + window]
center_pivot = pivot[center_cols]
fig, ax = plt.subplots(figsize=(10, 2.4))
if sns is not None:
    sns.heatmap(center_pivot, cmap="coolwarm", center=0, vmin=-v_bound, vmax=v_bound, ax=ax, cbar_kws={"label": "Delta probability"})
else:
    im = ax.imshow(center_pivot.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-v_bound, vmax=v_bound)
    fig.colorbar(im, ax=ax, label="Delta probability")
    ax.set_yticks(range(4), ["A", "C", "G", "T"])
ax.set_xlabel("Position relative to candidate A")
ax.set_ylabel("Mutated base")
ax.set_xticks(np.arange(len(center_cols)) + 0.5)
ax.set_xticklabels([str(c - center) for c in center_cols], rotation=90)
fig.tight_layout()
save_figure(fig, fig4_dir / "fig4C_mutation_center_51nt_heatmap.png")

## 10. Figure 4D: Mutation-Based Sequence Logo

In [ ]:
if logomaker is None:
    raise ImportError("Install logomaker to draw sequence logos: pip install logomaker")

logo_df = mutation_only[(mutation_only["position0"] >= center - window) & (mutation_only["position0"] <= center + window)].copy()
logo_df["rel_position"] = logo_df["position0"] - center
logo_matrix = logo_df.pivot_table(values="Delta_pred_prob", index="rel_position", columns="mutated_base", aggfunc="mean")
logo_matrix = logo_matrix.reindex(index=np.arange(-window, window + 1), columns=["A", "C", "G", "T"]).fillna(0.0)
for pos in logo_matrix.index:
    max_abs = logo_matrix.loc[pos].abs().max()
    if max_abs > 0:
        logo_matrix.loc[pos] = logo_matrix.loc[pos] / max_abs

fig, ax = plt.subplots(figsize=(10, 3))
logomaker.Logo(logo_matrix, ax=ax, color_scheme="classic", shade_below=0, fade_below=0.5)
ax.set_ylabel("Mutation Effect")
ax.set_xlabel("Position relative to candidate A")
ax.set_xticks(np.arange(-window, window + 1))
ax.set_xticklabels(np.arange(-window, window + 1), rotation=90)
fig.tight_layout()
save_figure(fig, fig4_dir / "fig4D_mutation_logo.png")
logo_matrix.to_csv(fig4_dir / "fig4D_mutation_logo_matrix.csv")

## 11. Supplementary Figure S1: Training Cost by Input Length

Expected input table:

`outputs/metrics/length_benchmark/training_cost.csv`

Required columns: `length`, `time`, `gpu`. `time` can be seconds per step and `gpu` can be MB.

In [ ]:
figs_dir = ensure_dir(FIGURE_DIR / "supplementary")
TRAINING_COST_CSV = PROJECT_ROOT / "outputs" / "metrics" / "length_benchmark" / "training_cost.csv"
if TRAINING_COST_CSV.exists():
    cost_df = pd.read_csv(TRAINING_COST_CSV)
    if "length" not in cost_df.columns and "model" in cost_df.columns:
        cost_df["length"] = cost_df["model"].astype(str).str.extract(r"(\d+)").astype(int)
    cost_df = cost_df.sort_values("length")
    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(cost_df["length"].astype(str), cost_df["gpu"] / 1024, marker="o", color="blue", label="Video memory size")
    ax1.set_ylabel("Video memory size (GB)")
    ax2 = ax1.twinx()
    ax2.plot(cost_df["length"].astype(str), cost_df["time"], marker="s", linestyle="--", color="red", label="Training time")
    ax2.set_ylabel("Training time")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False)
    fig.tight_layout()
    save_figure(fig, figs_dir / "figS1_training_cost.png")
else:
    print(f"skip Figure S1; missing {TRAINING_COST_CSV}")

## 12. Supplementary Figure S3: Sequence Logos for Low-Performance Atlas Datasets

Edit `LOW_PERFORMANCE_CSVS` to include the independent test CSV files used in the paper discussion.

In [ ]:
LOW_PERFORMANCE_CSVS = {
    # "HEK293T_m6A_label_seq": PROJECT_ROOT / "data" / "processed" / "independent_test_sets" / "atlas" / "full" / "human" / "HEK293T_m6A-label-seq__merged.csv",
    # "CRMM1_miCLIP": PROJECT_ROOT / "data" / "processed" / "independent_test_sets" / "atlas" / "full" / "human" / "CRMM1_miCLIP__merged.csv",
}


def sequence_frequency_matrix(sequences):
    sequences = [str(s).upper() for s in sequences if isinstance(s, str)]
    if not sequences:
        raise ValueError("No sequences provided")
    length = len(sequences[0])
    sequences = [s for s in sequences if len(s) == length]
    rows = []
    for pos in range(length):
        total = len(sequences)
        rows.append({base: sum(seq[pos] == base for seq in sequences) / total for base in ["A", "C", "G", "T"]})
    return pd.DataFrame(rows)


def plot_center_sequence_logo(csv_path, label_value=1, name="dataset"):
    if logomaker is None:
        raise ImportError("Install logomaker to draw sequence logos: pip install logomaker")
    df = pd.read_csv(csv_path)
    seqs = df.loc[df["label"].astype(int) == int(label_value), "seq"].astype(str).tolist()
    freq = sequence_frequency_matrix(seqs)
    center = len(freq) // 2
    sub = freq.iloc[center - 25 : center + 26].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(10, 3))
    logomaker.Logo(sub[["A", "C", "G", "T"]], ax=ax, color_scheme="classic")
    ax.set_xticks(np.arange(51))
    ax.set_xticklabels(np.arange(-25, 26), rotation=90)
    ax.set_ylabel("Frequency")
    ax.set_xlabel("Position relative to candidate A")
    fig.tight_layout()
    save_figure(fig, figs_dir / f"figS3_{name}_label{label_value}_logo.png")

for name, csv_path in LOW_PERFORMANCE_CSVS.items():
    if Path(csv_path).exists():
        plot_center_sequence_logo(csv_path, label_value=1, name=name)
    else:
        print(f"skip missing logo input: {csv_path}")